# osmo-minioperator — refaire la campagne de tests du README

Ce notebook rejoue la section **Tests** du `README.md` :

1. **construction de zéro** de l'image (`docker build --no-cache`) et contrôle de la ROM DSP ;
2. **verdict à `INSNS=60000`** (ancien défaut) : attendu *échec de la voix montante* ;
3. **verdict à `INSNS=120000`** (défaut actuel) : attendu *OK 2/2*.

Puis il met les résultats en tableau, les compare à ceux du README et trace les
enregistrements (`/tmp/verif/appel*.raw`, s16le 8 kHz mono) : annonce descendante
et retour du 1 kHz par l'écho du 600.

**Prérequis** : hôte Linux avec Docker (lancé en root ou membre du groupe `docker`),
PulseAudio hôte (`/run/pulse/native`) — sans lui l'appel est muet côté hôte mais `verif.sh`
mesure quand même la voix. Durée : ~4 min de build + ~4 min par verdict.
L'annonce et le 1 kHz sortent dans les haut-parleurs de l'hôte.

**En CLI, enregistré avec asciinema** : `./tests-cli.sh` (voir l'en-tête du script).

In [ ]:
import os, re, subprocess, time, shlex, platform, glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO     = Path(os.environ.get("REPO", Path.cwd()))
IMG      = os.environ.get("IMG", "osmo-minioperator")   # image construite et testée
NO_CACHE = os.environ.get("NO_CACHE", "1") == "1"   # 0 : build incrémental (bien plus rapide)
BUILD    = os.environ.get("BUILD", "1") == "1"      # 0 : réutiliser l'image IMG existante
APPELS   = int(os.environ.get("APPELS", "2"))       # appels au 600 par verdict
CAMPAGNES = [60000, 120000]          # valeurs d'INSNS testées
OUT      = REPO / "tests-out"        # enregistrements récupérés de chaque verdict

# En notebook : rendu Markdown et figures en ligne. En CLI (ipython, asciinema) : texte brut,
# figures enregistrées en PNG dans OUT.
NOTEBOOK = type(get_ipython()).__name__ == "ZMQInteractiveShell"
if not NOTEBOOK:
    plt.switch_backend("Agg")
OUT.mkdir(parents=True, exist_ok=True)

def afficher(texte):
    display(Markdown(texte)) if NOTEBOOK else print(texte)

def montrer(nom):
    plt.savefig(OUT / f"{nom}.png", dpi=110)
    plt.show() if NOTEBOOK else print(f"[figure] {OUT / (nom + '.png')}")
    plt.close()

ANSI = re.compile(r"\x1b\[[0-9;]*m")

def sh(cmd, cwd=REPO, show=True):
    # Lance une commande shell, affiche la sortie au fil de l'eau, renvoie (code, sortie, durée).
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, bufsize=1)
    lignes = []
    for l in p.stdout:
        l = ANSI.sub("", l.rstrip("\n")); lignes.append(l)
        if show: print(l, flush=True)
    return p.wait(), "\n".join(lignes), time.time() - t0

## 0. Contexte de la campagne

In [ ]:
_, head, _ = sh("git rev-parse --short HEAD && git status --short | head", show=False)
_, dv, _   = sh("docker version --format '{{.Server.Version}}'", show=False)
pulse = next((s for s in ("/run/pulse/native", "/var/run/pulse/native") if Path(s).is_socket()), None)
afficher(f'''
| | |
|---|---|
| date | {time.strftime("%Y-%m-%d %H:%M")} |
| commit | `{head.splitlines()[0]}` {"(arbre modifié)" if len(head.splitlines()) > 1 else ""} |
| hôte | {platform.platform()} |
| Docker | {dv} |
| PulseAudio hôte | {pulse or "**absent** (appels muets côté hôte)"} |
| image | `{IMG}` |
''')

## 1. Construction de zéro

README : `sudo docker build -t osmo-minioperator . --no-cache` (250 s), avec les deux
contrôles de la ROM DSP qui passent.

In [ ]:
if BUILD:
    rc, build_log, dt = sh(f"docker build --progress=plain -t {IMG} {'--no-cache' if NO_CACHE else ''} .")
    print(f"\n==> code {rc}, {dt:.0f} s")
    assert rc == 0, "échec du build"
else:
    build_log, dt = "", 0
    print("build sauté (BUILD=False)")

In [ ]:
attendus = ["somme de controle verifiee (SHA256SUMS.3606)",
            "ROM DSP 3606 ecrite dans /opt/GSM",
            "ROM DSP 3606 ecrite dans /opt/GSM/c54x_exe/rom",
            "/opt/GSM/calypso_dsp.txt: OK"]
if BUILD:
    for a in attendus:
        print(("OK     " if a in build_log else "ABSENT ") + a)
    assert all(a in build_log for a in attendus), "contrôle ROM DSP manquant dans le build"

## 2–3. Verdicts `verif.sh` à `INSNS=60000` puis `INSNS=120000`

Chaque verdict tourne dans un **conteneur neuf** (`--rm`), comme la commande du README :
`verif.sh` lance `run-dsp.sh` (qui lit `INSNS` dans l'environnement), attend le LU,
puis passe les appels au 600. `/tmp/verif` est monté sur l'hôte pour récupérer les
enregistrements. Pas de `-it` : le notebook n'a pas de terminal.

In [ ]:
def docker_args():
    a = ["--rm", "--init", "--cap-add", "NET_ADMIN", "--cap-add", "SYS_NICE", "--device", "/dev/net/tun"]
    if pulse:
        a += ["-v", f"{os.path.realpath(pulse)}:/run/pulse/native"]
    if Path("/dev/snd").is_dir():
        a += ["--device", "/dev/snd"]
    return a

LIGNE = re.compile(r"appel (\d+) : (OK|ECHEC)\s+\| etabli (oui|NON).*?\| voix DL (oui|NON) \(RMS max (\d+)\)"
                   r" \| voix UL \(echo 1 kHz\) (oui|NON) \(part 1 kHz ([\d.]+)\)")

def verdict(insns):
    d = OUT / f"insns-{insns}"; d.mkdir(parents=True, exist_ok=True)
    for f in d.glob("*.raw"): f.unlink()
    cmd = ["docker", "run", *docker_args(), "-e", f"INSNS={insns}", "-v", f"{d}:/tmp/verif",
           IMG, "./verif.sh", "--appels", str(APPELS)]
    print("$", " ".join(map(shlex.quote, cmd)), "\n")
    rc, log, dt = sh(" ".join(map(shlex.quote, cmd)))
    appels = [dict(insns=insns, appel=int(m[1]), ok=m[2] == "OK", etabli=m[3] == "oui",
                   dl=m[4] == "oui", rms=int(m[5]), ul=m[6] == "oui", part1k=float(m[7]))
              for m in LIGNE.finditer(log)]
    print(f"\n==> code {rc}, {dt:.0f} s, {len(appels)} appel(s) analysé(s)")
    return dict(insns=insns, rc=rc, duree=dt, log=log, appels=appels, dir=d)

resultats = {}

In [ ]:
resultats[60000] = verdict(60000)

In [ ]:
resultats[120000] = verdict(120000)

## 4. Comparaison avec le README

Attendu (README, campagne du 2026-10-29) :

| INSNS | établi | voix DL | voix UL | verdict |
|---|---|---|---|---|
| 60000 | oui | oui (RMS ~21000–25000) | **NON** (part 1 kHz 0.01) | ECHEC |
| 120000 | oui | oui (RMS ~4300–4500) | oui (part 1 kHz 0.82) | OK |

In [ ]:
ATTENDU = {60000: dict(etabli=True, dl=True, ul=False, rc=1),
           120000: dict(etabli=True, dl=True, ul=True, rc=0)}
oui = lambda b: "oui" if b else "**NON**"
rows = ["| INSNS | appel | établi | voix DL (RMS max) | voix UL (part 1 kHz) | verdict | conforme au README |",
        "|---|---|---|---|---|---|---|"]
conforme = True
for insns, r in resultats.items():
    att = ATTENDU.get(insns)
    if not r["appels"]:
        rows.append(f"| {insns} | — | pas d'appel (code {r['rc']}) | | | ECHEC | {'non' if att else '?'} |")
        conforme &= att is None
        continue
    for a in r["appels"]:
        ok_att = att is None or all(a[k] == att[k] for k in ("etabli", "dl", "ul"))
        conforme &= ok_att
        rows.append(f"| {insns} | {a['appel']} | {oui(a['etabli'])} | {oui(a['dl'])} ({a['rms']}) | "
                    f"{oui(a['ul'])} ({a['part1k']:.2f}) | {'OK' if a['ok'] else 'ECHEC'} | "
                    f"{'oui' if ok_att else '**non**'} |")
    if att: conforme &= (r["rc"] == 0) == (att["rc"] == 0)
afficher("\n".join(rows))
afficher(f"**Campagne {'conforme' if conforme else 'NON conforme'} au README.**")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, cle, seuil, titre in ((ax1, "rms", 300, "Voix descendante : RMS max (seuil 300)"),
                              (ax2, "part1k", 0.4, "Voix montante : part du 1 kHz dans l'écho (seuil 0,4)")):
    x, etiquettes = 0, []
    for insns, r in resultats.items():
        for a in r["appels"]:
            ax.bar(x, a[cle], color="#2a7ab0" if a[cle] > seuil else "#c0392b")
            etiquettes.append(f"{insns // 1000}k\nappel {a['appel']}"); x += 1
    ax.axhline(seuil, color="k", ls="--", lw=1)
    ax.set_xticks(range(len(etiquettes)), etiquettes); ax.set_title(titre, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); montrer("mesures")

## 5. Enregistrements

`gsm_audio.monitor` (ce qui sort du vocodeur ROM du mobile) pendant chaque appel :
l'annonce d'Asterisk, puis le 1 kHz injecté dans `gsm_mic` et renvoyé par l'écho du 600.
Une raie à 1 kHz dans la seconde moitié du spectrogramme = voix montante OK.

In [ ]:
TAUX = 8000
fichiers = [(insns, f) for insns, r in resultats.items() for f in sorted(r["dir"].glob("appel*.raw"))]
if not fichiers:
    print("aucun enregistrement récupéré")
else:
    fig, axes = plt.subplots(len(fichiers), 2, figsize=(12, 2.6 * len(fichiers)), squeeze=False)
    for (insns, f), (a1, a2) in zip(fichiers, axes):
        s = np.fromfile(f, dtype="<i2").astype(float)
        t = np.arange(len(s)) / TAUX
        a1.plot(t, s, lw=0.3, color="#2a7ab0"); a1.set_xlim(0, t[-1] if len(t) else 1)
        a1.set_title(f"INSNS={insns} — {f.name} ({len(s) / TAUX:.0f} s)", fontsize=9)
        a1.set_xlabel("s"); a1.spines[["top", "right"]].set_visible(False)
        if len(s) > 512:
            a2.specgram(s, NFFT=256, Fs=TAUX, noverlap=128, cmap="magma")
            a2.axhline(1000, color="c", ls=":", lw=0.8)
        a2.set_title("spectrogramme (pointillé : 1 kHz)", fontsize=9); a2.set_xlabel("s"); a2.set_ylabel("Hz")
    plt.tight_layout(); montrer("enregistrements")

## 6. Verdict de la campagne

In [ ]:
for insns, r in resultats.items():
    print(f"INSNS={insns:>6} : code {r['rc']}, {r['duree']:.0f} s, "
          + ", ".join(f"appel {a['appel']} {'OK' if a['ok'] else 'ECHEC'}" for a in r["appels"]))
assert resultats[120000]["rc"] == 0, "le défaut INSNS=120000 doit donner un verdict OK"
print("\nCampagne", "conforme" if conforme else "NON conforme", "au README.")